# 07 - Adversarial Robustness Testing

Tests how easily the Autoencoder can be evaded by an attacker who deliberately perturbs traffic to look more "normal." Uses SHAP findings to craft a targeted attack against the model's top-weighted features (Packet Length Variance, FIN Flag Count, Bwd Packet Length Std), compared against a random-noise baseline attack.

#### Load model, data, and the correctly-flagged anomalies (reuse from notebook 06)

In [1]:
import torch, joblib, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('/home/vboxuser/FEAR/data/cicids2017_cleaned.csv')
X = df.drop(columns=['Label', 'Binary_Label'])
y = df['Binary_Label']
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = joblib.load('/home/vboxuser/FEAR/models/scaler.pkl')
X_test_scaled = scaler.transform(X_test)
feature_names = X.columns.tolist()

#### Rebuild model and get baseline predictions

In [3]:
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, input_dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = Autoencoder(input_dim=X_test_scaled.shape[1])
model.load_state_dict(torch.load('/home/vboxuser/FEAR/models/autoencoder.pt'))
model.eval()

X_test_tensor = torch.FloatTensor(X_test_scaled)
with torch.no_grad():
    reconstructions = model(X_test_tensor)
    reconstruction_errors = torch.mean((X_test_tensor - reconstructions) ** 2, dim=1).numpy()

threshold = np.percentile(reconstruction_errors[y_test.values == 0], 80)
preds_binary = (reconstruction_errors > threshold).astype(int)

anomaly_indices = np.where((y_test.values == 1) & (preds_binary == 1))[0]
print(f'Correctly flagged anomalies to attack: {len(anomaly_indices)}')

Correctly flagged anomalies to attack: 101068


#### Sample anomalies to attack, find benign reference values

In [4]:
np.random.seed(42)
attack_sample_idx = np.random.choice(anomaly_indices, size=2000, replace=False)
X_attack_original = X_test_scaled[attack_sample_idx].copy()

# Benign reference: mean values in scaled space, what "normal" looks like to the model
benign_mask = y_test.values == 0
X_benign_scaled = X_test_scaled[benign_mask]
benign_mean = X_benign_scaled.mean(axis=0)

print(f'Attacking {len(attack_sample_idx)} correctly-flagged anomalies')

Attacking 2000 correctly-flagged anomalies


#### Targeted attack: perturb only the top SHAP-driving features

In [5]:
top_features = ['Packet Length Variance', 'FIN Flag Count', 'Bwd Packet Length Std']
top_feature_idx = [feature_names.index(f) for f in top_features]

X_attack_targeted = X_attack_original.copy()
perturbation_strength = 0.8  # move 80% of the way toward the benign mean

for idx in top_feature_idx:
    X_attack_targeted[:, idx] = X_attack_original[:, idx] + \
        perturbation_strength * (benign_mean[idx] - X_attack_original[:, idx])

print('Targeted attack crafted (3 features perturbed)')

Targeted attack crafted (3 features perturbed)


#### Random attack: equal-magnitude noise across all features 

In [6]:
np.random.seed(42)
noise_scale = 0.5  # similar order of magnitude to the targeted shift
random_noise = np.random.normal(0, noise_scale, X_attack_original.shape)
X_attack_random = X_attack_original + random_noise

print('Random attack crafted (all 78 features perturbed)')

Random attack crafted (all 78 features perturbed)


In [7]:
def evaluate_attack(X_attacked, label):
    X_tensor = torch.FloatTensor(X_attacked)
    with torch.no_grad():
        recon = model(X_tensor)
        errors = torch.mean((X_tensor - recon) ** 2, dim=1).numpy()
    still_detected = (errors > threshold).sum()
    evasion_rate = 1 - (still_detected / len(X_attacked))
    print(f'{label}: {still_detected}/{len(X_attacked)} still detected ({still_detected/len(X_attacked)*100:.1f}% recall) | Evasion rate: {evasion_rate*100:.1f}%')
    return errors

print('--- Original (before any attack) ---')
_ = evaluate_attack(X_attack_original, 'Original')

print('\n--- Targeted attack (top 3 SHAP features) ---')
errors_targeted = evaluate_attack(X_attack_targeted, 'Targeted')

print('\n--- Random attack (noise across all 78 features) ---')
errors_random = evaluate_attack(X_attack_random, 'Random')

--- Original (before any attack) ---
Original: 2000/2000 still detected (100.0% recall) | Evasion rate: 0.0%

--- Targeted attack (top 3 SHAP features) ---
Targeted: 2000/2000 still detected (100.0% recall) | Evasion rate: 0.0%

--- Random attack (noise across all 78 features) ---
Random: 2000/2000 still detected (100.0% recall) | Evasion rate: 0.0%


#### Check the actual error values, not just pass/fail

In [8]:
print('Original mean error:', reconstruction_errors[attack_sample_idx].mean())
print('Targeted attack mean error:', errors_targeted.mean())
print('Random attack mean error:', errors_random.mean())
print('Threshold:', threshold)

Original mean error: 2.3705
Targeted attack mean error: 0.7307848
Random attack mean error: 2.692879
Threshold: 0.029812155


#### Stronger targeted attack: fully replace top features (100% instead of 80%), and expand to top 10 features

In [9]:
top_10_features = importance_df['feature'].tolist()[:10] if 'importance_df' in dir() else top_features
top_10_idx = [feature_names.index(f) for f in top_10_features]

X_attack_strong = X_attack_original.copy()
for idx in top_10_idx:
    X_attack_strong[:, idx] = benign_mean[idx]  # full replacement, not partial

errors_strong = evaluate_attack(X_attack_strong, 'Strong targeted (top 10, full replacement)')

Strong targeted (top 10, full replacement): 2000/2000 still detected (100.0% recall) | Evasion rate: 0.0%


#### Stronger random attack: larger noise scale

In [11]:
for scale in [1.0, 2.0, 5.0]:
    np.random.seed(42)
    noise = np.random.normal(0, scale, X_attack_original.shape)
    X_random_scaled = X_attack_original + noise
    print(f'\nNoise scale {scale}:')
    _ = evaluate_attack(X_random_scaled, f'Random (scale={scale})')


Noise scale 1.0:
Random (scale=1.0): 2000/2000 still detected (100.0% recall) | Evasion rate: 0.0%

Noise scale 2.0:
Random (scale=2.0): 2000/2000 still detected (100.0% recall) | Evasion rate: 0.0%

Noise scale 5.0:
Random (scale=5.0): 2000/2000 still detected (100.0% recall) | Evasion rate: 0.0%


#### Brake the Ice

In [12]:
X_full_replace = np.tile(benign_mean, (2000, 1))  # every single feature = benign average
errors_full = evaluate_attack(X_full_replace, 'Full replacement (all 78 features = benign mean)')

Full replacement (all 78 features = benign mean): 2000/2000 still detected (100.0% recall) | Evasion rate: 0.0%


In [14]:
print('Full replacement mean error:', errors_full.mean())
print('Threshold:', threshold)
print('Min original anomaly error:', reconstruction_errors[attack_sample_idx].min())
print('Benign mean error (for comparison):', reconstruction_errors[y_test.values==0].mean())

Full replacement mean error: 0.041586157
Threshold: 0.029812155
Min original anomaly error: 0.02986765
Benign mean error (for comparison): 0.046952706


#### White-box attack / Test the ceiling 

In [15]:
# Gradient-based adversarial attack: find minimal perturbation that minimizes reconstruction error
X_adv = torch.FloatTensor(X_attack_original.copy()).requires_grad_(True)
optimizer_adv = torch.optim.Adam([X_adv], lr=0.01)

for step in range(200):
    optimizer_adv.zero_grad()
    recon = model(X_adv)
    error = torch.mean((X_adv - recon) ** 2, dim=1)
    loss = error.mean()  # minimize reconstruction error = minimize anomaly score
    loss.backward()
    optimizer_adv.step()

with torch.no_grad():
    recon_final = model(X_adv)
    errors_gradient_attack = torch.mean((X_adv - recon_final) ** 2, dim=1).numpy()

still_detected = (errors_gradient_attack > threshold).sum()
print(f'Gradient attack: {still_detected}/{len(X_adv)} still detected ({still_detected/len(X_adv)*100:.1f}%)')
print(f'Mean error after gradient attack: {errors_gradient_attack.mean():.4f}')

Gradient attack: 1294/2000 still detected (64.7%)
Mean error after gradient attack: 0.6983
